In [1]:
import os
import glob
import json

from pathlib import Path
from typing import Generator, Tuple, Optional, List, Tuple, Any

import numpy as np
import cv2
from PIL import Image


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

In [20]:
from ultralytics import YOLO
import numpy as np
import torch
import cv2
import torch.nn.functional as F

from ultralytics import YOLO

torch.cuda.empty_cache()

# Load a model
model = YOLO("yolo11n.pt")  # load an official model

In [ ]:
# Predict with the model
results = model.predict("https://ultralytics.com/images/bus.jpg")  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [
        result.names[cls.item()] for cls in result.boxes.cls.int()
    ]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box

In [18]:
from pathlib import Path

import cv2
import torch
from effdet import create_model
from effdet.config import get_efficientdet_config
from torchvision import transforms

from boxmot import BotSort
from boxmot.utils.ops import letterbox

# Load EfficientDet model
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = (
    "tf_efficientdet_d0"  # You can choose a different variant like 'tf_efficientdet_d3'
)
config = get_efficientdet_config(model_name)
effdet_model = create_model(model_name, bench_task="predict", pretrained=True).to(
    device
)
effdet_model.eval()

DetBenchPredict(
  (model): EfficientDet(
    (backbone): EfficientNetFeatures(
      (conv_stem): Conv2dSame(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn1): BatchNormAct2d(
        32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): SiLU(inplace=True)
      )
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (bn1): BatchNormAct2d(
              32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): SiLU(inplace=True)
            )
            (aa): Identity()
            (se): SqueezeExcite(
              (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (act1): SiLU(inplace=True)
              (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1

In [24]:
# Initialize the tracker
tracker = BotSort(
    reid_weights=Path("osnet_x0_25_msmt17.pt"),  # Path to ReID model
    device=0,  # Use CPU for inference
    half=False,
)

# input_size = config.image_size

# preprocess = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF
To: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/notebooks/osnet_x0_25_msmt17.pt
100%|██████████| 3.06M/3.06M [00:00<00:00, 7.08MB/s]


In [1]:
def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

In [6]:
SAMPLE_PATH = DATA_DIR / "public_test" / "samples" / "BlackBox_0"
IMAGE_PATH = SAMPLE_PATH / "object_images"

images = glob.glob(str(IMAGE_PATH / "*.jpg"))
video_path = SAMPLE_PATH / "drone_video.mp4"

In [2]:
import json
import cv2
import os
from pathlib import Path
import re


def extract_yolo_frames(
    annotation_file: str, samples_root: str, output_root: str = "yolo_dataset"
):
    """
    Convert structured video annotations into YOLO training dataset:
    - Save FULL FRAME (not cropped)
    - Save YOLO (.txt) labels with normalized bounding boxes
    - Automatically infer class ID from video_id prefix (e.g., Backpack_0 → Backpack)

    Folder layout expected:
    observing/train/
        annotations/annotations.json
        samples/
            Backpack_0/drone_video.mp4
            Backpack_1/drone_video.mp4
            Jacket_0/drone_video.mp4
            ...

    YOLO output:
        yolo_dataset/
            images/
            labels/
    """

    annotation_file = Path(annotation_file)
    samples_root = Path(samples_root)
    output_root = Path(output_root)

    out_img = output_root / "images"
    out_lbl = output_root / "labels"

    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    with open(annotation_file, "r") as f:
        data = json.load(f)

    print(f"[INFO] Loaded {len(data)} annotation groups.")

    # ---------------------------------------------------------
    # 1. Create class mapping automatically
    # ---------------------------------------------------------
    class_names = set()

    for item in data:
        video_id = item["video_id"]
        class_name = re.sub(r"_\d+$", "", video_id)  # Backpack_0 → Backpack
        class_names.add(class_name)

    class_names = sorted(list(class_names))
    class_map = {name: idx for idx, name in enumerate(class_names)}

    print("[INFO] Class mapping:")
    for k, v in class_map.items():
        print(f"  {k:15} → {v}")

    # ---------------------------------------------------------
    # 2. Process each annotated video
    # ---------------------------------------------------------
    for item in data:
        video_id = item["video_id"]
        annotations = item["annotations"]

        class_name = re.sub(r"_\d+$", "", video_id)
        class_id = class_map[class_name]

        video_path = samples_root / video_id / "drone_video.mp4"
        if not video_path.exists():
            print(f"[WARN] Missing video: {video_path}")
            continue

        print(f"[INFO] Processing: {video_id}")

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"[ERROR] Cannot open video: {video_path}")
            continue

        frame_cache = {}  # avoid re-reading same frame

        for ann in annotations:
            for bbox in ann.get("bboxes", []):
                fidx = bbox["frame"]

                # -----------------------------------------------------
                # Read full frame
                # -----------------------------------------------------
                if fidx not in frame_cache:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
                    ret, frame = cap.read()
                    if not ret:
                        print(f"[WARN] Failed reading frame {fidx} in {video_id}")
                        continue
                    frame_cache[fidx] = frame
                else:
                    frame = frame_cache[fidx]

                h, w = frame.shape[:2]

                # -----------------------------------------------------
                # Normalize YOLO coordinates from bbox
                # -----------------------------------------------------
                x1, y1, x2, y2 = bbox["x1"], bbox["y1"], bbox["x2"], bbox["y2"]

                cx = (x1 + x2) / 2 / w
                cy = (y1 + y2) / 2 / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                # -----------------------------------------------------
                # Save frame as image
                # -----------------------------------------------------
                img_name = f"{video_id}_frame{fidx}.jpg"
                img_path = out_img / img_name
                cv2.imwrite(str(img_path), frame)

                # -----------------------------------------------------
                # Save YOLO label
                # -----------------------------------------------------
                label_path = out_lbl / img_name.replace(".jpg", ".txt")
                with open(label_path, "a") as lf:
                    lf.write(f"{class_id} {cx} {cy} {bw} {bh}\n")

        cap.release()

    print("\n✔ DONE! YOLO dataset is ready.")
    print(f"Images saved to: {out_img}")
    print(f"Labels saved to: {out_lbl}")

In [3]:
extract_yolo_frames(
    annotation_file="/home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/observing/train/annotations/annotations.json",
    samples_root="/home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/observing/train/samples",
)

[INFO] Loaded 14 annotation groups.
[INFO] Class mapping:
  Backpack        → 0
  Jacket          → 1
  Laptop          → 2
  Lifering        → 3
  MobilePhone     → 4
  Person1         → 5
  WaterBottle     → 6
[INFO] Processing: Backpack_0
[INFO] Processing: Backpack_1
[INFO] Processing: Jacket_0
[INFO] Processing: Jacket_1
[INFO] Processing: Laptop_0
[INFO] Processing: Laptop_1
[INFO] Processing: Lifering_0
[INFO] Processing: Lifering_1
[INFO] Processing: MobilePhone_0
[INFO] Processing: MobilePhone_1
[INFO] Processing: Person1_0
[INFO] Processing: Person1_1
[INFO] Processing: WaterBottle_0
[INFO] Processing: WaterBottle_1

✔ DONE! YOLO dataset is ready.
Images saved to: yolo_dataset/images
Labels saved to: yolo_dataset/labels


In [2]:
def load_json(
    filepath: Path | str
) -> Optional[dict]:
    """Load and return JSON data from a file."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {filepath}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {filepath}: {e}")
        return None

In [5]:
annotations = load_json(
    DATA_DIR / "train" / "annotations" / "annotations.json"
)
annotations[0]

{'video_id': 'Backpack_0',
 'annotations': [{'bboxes': [{'frame': 3483,
     'x1': 321,
     'y1': 0,
     'x2': 381,
     'y2': 12},
    {'frame': 3484, 'x1': 302, 'y1': 0, 'x2': 387, 'y2': 21},
    {'frame': 3485, 'x1': 314, 'y1': 0, 'x2': 401, 'y2': 40},
    {'frame': 3486, 'x1': 325, 'y1': 0, 'x2': 412, 'y2': 49},
    {'frame': 3487, 'x1': 335, 'y1': 10, 'x2': 422, 'y2': 58},
    {'frame': 3488, 'x1': 345, 'y1': 21, 'x2': 431, 'y2': 70},
    {'frame': 3489, 'x1': 351, 'y1': 30, 'x2': 437, 'y2': 77},
    {'frame': 3490, 'x1': 359, 'y1': 46, 'x2': 446, 'y2': 95},
    {'frame': 3491, 'x1': 365, 'y1': 57, 'x2': 452, 'y2': 106},
    {'frame': 3492, 'x1': 370, 'y1': 67, 'x2': 457, 'y2': 116},
    {'frame': 3493, 'x1': 377, 'y1': 77, 'x2': 463, 'y2': 127},
    {'frame': 3494, 'x1': 379, 'y1': 85, 'x2': 465, 'y2': 134},
    {'frame': 3495, 'x1': 380, 'y1': 100, 'x2': 466, 'y2': 148},
    {'frame': 3496, 'x1': 386, 'y1': 109, 'x2': 471, 'y2': 157},
    {'frame': 3497, 'x1': 388, 'y1': 117, 

In [ ]:
sample = annotations[0]
video_id = sample["video_id"]
vid_annotations = sample["annotations"]

samples_path = DATA_DIR / "train" / "samples" / video_id

video_sample = samples_path / "object_images"
video_path = samples_path / "drone_video.mp4"

cv2.VideoCapture(str(video_path))

for ann in vid_annotations:
    for bbox in ann.get("bboxes", []):
        print(bbox)

In [17]:
len(annotations)

14

In [20]:
import cv2
import numpy as np
import pathlib
from PIL import Image, ImageSequence


output_dir = DATA_DIR / "output"
output_dir.mkdir(exist_ok=True)

for idx, item in enumerate(annotations):
    print(f"{idx}: {item['video_id']}")
    sample = annotations[idx]
    video_id = sample["video_id"]
    samples_path = DATA_DIR / "train" / "samples" / video_id

    video_path = samples_path / "drone_video.mp4"
    output_path = output_dir / f"drone_video_{video_id}_with_bboxes.gif"

    # Organize the tracking coordinates
    bboxes_list = sample["annotations"][0]["bboxes"]
    bbox_lookup = {item["frame"]: item for item in bboxes_list}

    start_frame = min(bbox_lookup.keys())
    end_frame = max(bbox_lookup.keys())

    # --- Open the Input Video (.mp4) ---
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Error: Could not open the video file at {video_path}")
        exit()

    # Get the frame-rate from the video to calculate accurate GIF frame duration
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25  # Fallback if metadata is missing
    duration = int(1000 / fps)  # Convert FPS to milliseconds per frame for PIL

    # Jump the video reader directly to the start frame of your bounding boxes
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frame_idx = start_frame

    processed_frames = []
    print(f"Processing MP4 frames {start_frame} to {end_frame}...")

    # --- Process the Frames ---
    while frame_idx <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break  # Video ended unexpectedly or reached the end

        # Draw the bounding box if it exists for this frame index
        if frame_idx in bbox_lookup:
            bbox = bbox_lookup[frame_idx]
            x1, y1, x2, y2 = bbox["x1"], bbox["y1"], bbox["x2"], bbox["y2"]

            # Draw the box (Green: (0, 255, 0), thickness: 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Label the frame number
            cv2.putText(
                frame,
                f"Frame: {frame_idx}",
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
            )

        # OpenCV operates in BGR, but PIL saves GIFs in RGB format
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        processed_pil_frame = Image.fromarray(frame_rgb)
        processed_frames.append(processed_pil_frame)

        frame_idx += 1

    # Clean up the OpenCV reader resource
    cap.release()

    # --- Save out as an efficient, looping GIF ---
    if processed_frames:
        print(f"Saving tracking snippet GIF to: {output_path}")
        # Ensure the output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        processed_frames[0].save(
            output_path,
            save_all=True,
            append_images=processed_frames[1:],
            duration=duration,
            loop=0  # Infinite looping
        )
        print("Done!")
    else:
        print("No frames were captured. Make sure the video contains the frames specified in your bboxes.")

0: Backpack_0
Processing MP4 frames 3483 to 3762...
Saving tracking snippet GIF to: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/output/drone_video_Backpack_0_with_bboxes.gif
Done!
1: Backpack_1
Processing MP4 frames 2524 to 3272...
Saving tracking snippet GIF to: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/output/drone_video_Backpack_1_with_bboxes.gif
Done!
2: Jacket_0
Processing MP4 frames 704 to 776...
Saving tracking snippet GIF to: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/output/drone_video_Jacket_0_with_bboxes.gif
Done!
3: Jacket_1
Processing MP4 frames 451 to 566...
Saving tracking snippet GIF to: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/output/drone_video_Jacket_1_with_bboxes.gif
Done!
4: Laptop_0
Processing MP4 frames 2594 to 2805...
Saving tracking snippet GIF to: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/output/drone_video_Laptop_0_with_bboxes.gif
Do